#### SQL and Pandas

In [3]:
from pathlib import Path
import sys

sys.path.append(str(Path("code").resolve()))
from sql_demo import build_demo

conn, cur, orders, customers = build_demo()

orders

,order_id,customer_id,amount,order_date
0,1,101,120.5,2024-01-02
1,2,102,75.0,2024-01-03
2,3,103,210.0,2024-01-04
3,4,101,55.0,2024-01-05
4,5,104,320.0,2024-01-06


In [ ]:
# Selecting columns SQL

sql_select = """ SELECT order_id, amount FROM orders;"""

cur.execute(sql_select). fetchall()

[(1, 120.5), (2, 75.0), (3, 210.0), (4, 55.0), (5, 320.0)]

In [5]:
# Pandas
orders[["order_id", "amount"]]

,order_id,amount
0,1,120.5
1,2,75.0
2,3,210.0
3,4,55.0
4,5,320.0


In [6]:
# Filtering SQL

sql_filter="""SELECT order_id, amount FROM orders WHERE amount > 100;"""
cur.execute(sql_filter). fetchall()

[(1, 120.5), (3, 210.0), (5, 320.0)]

In [7]:
# Pandas
large_orders = orders[orders["amount"] > 100]
large_orders[["order_id", "amount"]]

,order_id,amount
0,1,120.5
2,3,210.0
4,5,320.0


In [9]:
# Grouping and aggregating SQL

sql_group = """SELECT 
c.country, 
SUM(o.amount) AS total_amount 
FROM orders AS o 
JOIN customers AS c 
ON o.customer_id = c.customer_id 
GROUP BY c.country 
ORDER BY total_amount DESC;"""

cur.execute(sql_group). fetchall()

[('US', 495.5), ('DE', 210.0), ('UK', 75.0)]

In [11]:
# Pandas

orders_with_customers = orders.merge(customers, on="customer_id", how="left")

country_summary = (
    orders_with_customers.groupby("country")["amount"]
    .agg(total_amount=("sum"))
    .sort_values("total_amount", ascending=False)
)

country_summary

,total_amount
country,
US,495.5
DE,210.0
UK,75.0


In [ ]:
''' Joins SQL When tables grow larger, 
performing the join in the database first and then loading the
joined result into pandas can be more efficient'''

sql_join = """SELECT 
o.order_id, 
o.amount, 
c.country 
FROM orders AS o 
LEFT JOIN customers AS c 
ON o.customer_id = c.customer_id;"""

cur.execute(sql_join). fetchall()

[(1, 120.5, 'US'),
 (2, 75.0, 'UK'),
 (3, 210.0, 'DE'),
 (4, 55.0, 'US'),
 (5, 320.0, 'US')]